# Thêm Thư Viện

In [38]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [39]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2024;'
)
conn_dwh_lib = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=dwh_lib;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2024;'
)


# ETL bảng Dim_Date

## Đọc dataset từ file CSV

In [4]:
df_data_date = pd.read_csv("./Source-Data/Data_Date.csv")
print(df_data_date)

       Date_key   Full_date                     Date_text   Day_name  \
0      19700101    1/1/1970     Thursday, January 1, 1970   Thursday   
1      19700102    1/2/1970       Friday, January 2, 1970     Friday   
2      19700103    1/3/1970     Saturday, January 3, 1970   Saturday   
3      19700104    1/4/1970       Sunday, January 4, 1970     Sunday   
4      19700105    1/5/1970       Monday, January 5, 1970     Monday   
...         ...         ...                           ...        ...   
29215  20491227  12/27/2049     Monday, December 27, 2049     Monday   
29216  20491228  12/28/2049    Tuesday, December 28, 2049    Tuesday   
29217  20491229  12/29/2049  Wednesday, December 29, 2049  Wednesday   
29218  20491230  12/30/2049   Thursday, December 30, 2049   Thursday   
29219  20491231  12/31/2049     Friday, December 31, 2049     Friday   

       Week_of_quarter  Day_of_week  Month  Quarter  Year  Day  
0                    1            4      1        1  1970    1  
1    

## Xử lý data

In [5]:
# Tạo hàng dữ liệu giả lập cho ngày không xác định
new_row = pd.DataFrame({
    'Date_key': [0],
    'Full_date': ['1/1/9999'],     # Đặt ngày giả định là 1900-01-01
    'Date_text': ['Unknown Date'],
    'Day': [0],
    'Week_of_quarter': [0],
    'Month': [0],
    'Quarter': [0],
    'Year': [9999],
    'Day_of_week': [0],
    'Day_name': ['Unknown']
})
df_data_date = pd.concat([df_data_date, new_row], ignore_index=True) # Thêm vào dataset
df_data_date = df_data_date.sort_values(by='Date_key', ascending=True).reset_index(drop=True) # sắp xếp lại cho dễ nhìn
print(df_data_date)

       Date_key   Full_date                     Date_text   Day_name  \
0             0    1/1/9999                  Unknown Date    Unknown   
1      19700101    1/1/1970     Thursday, January 1, 1970   Thursday   
2      19700102    1/2/1970       Friday, January 2, 1970     Friday   
3      19700103    1/3/1970     Saturday, January 3, 1970   Saturday   
4      19700104    1/4/1970       Sunday, January 4, 1970     Sunday   
...         ...         ...                           ...        ...   
29216  20491227  12/27/2049     Monday, December 27, 2049     Monday   
29217  20491228  12/28/2049    Tuesday, December 28, 2049    Tuesday   
29218  20491229  12/29/2049  Wednesday, December 29, 2049  Wednesday   
29219  20491230  12/30/2049   Thursday, December 30, 2049   Thursday   
29220  20491231  12/31/2049     Friday, December 31, 2049     Friday   

       Week_of_quarter  Day_of_week  Month  Quarter  Year  Day  
0                    0            0      0        0  9999    0  
1    

## Load data

### [Nếu cần] Clear bảng 

In [ ]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Date"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

### Load vào bảng DIM_Date

In [6]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Date (Date_key, Full_date, Date_text, Day, Week_of_quarter, Month, Quarter, Year, Day_of_week, Day_name)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                """
for index, row in df_data_date.iterrows():
    values = (row['Date_key'], 
              row['Full_date'], 
              row['Date_text'], 
              row['Day'], 
              row['Week_of_quarter'], 
              row['Month'], 
              row['Quarter'], 
              row['Year'], 
              row['Day_of_week'], 
              row['Day_name'])
    cursor_dwh.execute(insert_query, values)
    conn_dwh_lib.commit()

# ETL bảng Dim_Khoa

## Đọc data từ CSV

In [16]:
df_data_khoa = pd.read_csv("./Source-Data/Data_Khoa.csv")
df_data_khoa['Ten_khoa'] = df_data_khoa['Ten_khoa'].apply(lambda x: x.title() if isinstance(x, str) else x)
print(df_data_khoa)

    ID_Khoa                             Ten_khoa
0         1                    Lý Luận Chính Trị
1         2                    Khoa Học Ứng Dụng
2         3                   Cơ Khí Chế Tạo Máy
3         4                       Điện - Điện Tử
4         5                      Cơ Khí Động Lực
5         6                              Kinh Tế
6         7                  Công Nghệ Thông Tin
7         8                   In Và Truyền Thông
8         9          Công Nghệ May Và Thời Trang
9        10       Công Nghệ Hóa Học Và Thực Phẩm
10       11                             Xây Dựng
11       12                            Ngoại Ngữ
12       13               Đào Tạo Chất Lượng Cao
13       14                Viện Sư Phạm Kỹ Thuật
14       15  Trường Trung Học Kỹ Thuật Thực Hành


## Xử lý data

In [17]:
# Tạo hàng dữ liệu giả lập cho khoa không xác định
new_row = pd.DataFrame({'ID_Khoa': [0], 'Ten_khoa': ['(Không xác định)']})
df_data_khoa = pd.concat([df_data_khoa, new_row], ignore_index=True) # Thêm vào dataset
df_data_khoa = df_data_khoa.sort_values(by='ID_Khoa', ascending=True).reset_index(drop=True) # sắp xếp lại cho dễ nhìn
print(df_data_khoa)

    ID_Khoa                             Ten_khoa
0         0                     (Không xác định)
1         1                    Lý Luận Chính Trị
2         2                    Khoa Học Ứng Dụng
3         3                   Cơ Khí Chế Tạo Máy
4         4                       Điện - Điện Tử
5         5                      Cơ Khí Động Lực
6         6                              Kinh Tế
7         7                  Công Nghệ Thông Tin
8         8                   In Và Truyền Thông
9         9          Công Nghệ May Và Thời Trang
10       10       Công Nghệ Hóa Học Và Thực Phẩm
11       11                             Xây Dựng
12       12                            Ngoại Ngữ
13       13               Đào Tạo Chất Lượng Cao
14       14                Viện Sư Phạm Kỹ Thuật
15       15  Trường Trung Học Kỹ Thuật Thực Hành


## Load data

### [Nếu cần] Clear bảng

In [18]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Khoa"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

### Load data vào bảng Dim_Khoa

In [19]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Khoa (ID_khoa, Ten_khoa) 
                VALUES (?, ?)
                """
for index, row in df_data_khoa.iterrows():
    values = (row['ID_Khoa'], 
              row['Ten_khoa'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Dim_Dan_Toc

## Đọc data từ CSV

In [20]:
df_data_dantoc = pd.read_csv("./Source-Data/Data_Dan_toc.csv")
df_data_dantoc = df_data_dantoc.where(pd.notnull(df_data_dantoc), None)
print(df_data_dantoc)

    Mã               Tên                                       Tên gọi khác
0    1              Kinh                                               Việt
1    2               Tày          Thổ, Ngạn, Phén, Thù Lao, Pa Dí, Tày Khao
2    3              Thái  Tày Đăm, Tày Mười, Tày Thanh, Mán Thanh, Hàng ...
3    4               Hoa  Hán, Triều Châu, Phúc Kiến, Quảng Đông, Hải Na...
4    5            Khơ-me             Cur, Cul, Cu, Thổ, Việt gốc Miên, Krôm
5    6             Mường               Mol, Mual, Mọi, Mọi Bi, Ao Tá, Ậu Tá
6    7              Nùng  Xuồng, Giang, Nùng An, Phàn Sinh, Nùng Cháo, N...
7    8             HMông  Mèo, Hoa, Mèo Xanh, Mèo Đỏ, Mèo Đen, Ná Mẻo, M...
8    9               Dao  Mán, Động, Trại, Xá, Dìu, Miên, Kiềm, Miền, Qu...
9   10           Gia-rai    Giơ-rai, Tơ-buăn, Chơ-rai, Hơ-bau, Hđrung, Chor
10  11              Ngái                            Xín, Lê, Đản, Khách Gia
11  12              Ê-đê  Ra-đê, Đê, Kpạ, A-đham, Krung, Ktul, Đliê Ruê,...
12  13      

## Load data

### [Nếu cần] Clear bảng

In [21]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Dan_toc"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

### Load data vào bảng DIM_Dan_toc

In [22]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Dan_toc (ID_dan_toc, Dan_toc, Ten_khac) 
                VALUES (?, ?, ?)
                """
for index, row in df_data_dantoc.iterrows():
    values = (row['Mã'], 
              row['Tên'], 
              row['Tên gọi khác'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Dim_Trinh_do

## Đọc data từ SQL Server

In [23]:
query_trinhdo = "SELECT trinh_do_id, dbo.DecodeUTF8String(loai_trinh_do) AS Trinh_do FROM Trinh_do "
df_trinhdo = pd.read_sql(query_trinhdo, conn_libol)
print(df_trinhdo)

   trinh_do_id                 Trinh_do
0           10                   PGS.TS
1            3                 Cao đẳng
2            4                  Đại học
3            5                  Thạc sĩ
4            6                  Tiến sĩ
5            7              Phó tiến sĩ
6           11  Trung học chuyên nghiệp
7            9             Trung học PT
8           12                    12/12


C:\Users\admin\AppData\Local\Temp\ipykernel_21708\4146533011.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_trinhdo = pd.read_sql(query_trinhdo, conn_libol)


## Xử lý data

In [24]:
new_row = pd.DataFrame({'trinh_do_id': [0], 'Trinh_do': ['(Không xác định)']}) # Thêm 1 dòng không xác định
df_trinhdo = pd.concat([df_trinhdo, new_row], ignore_index=True)
df_trinhdo = df_trinhdo.sort_values(by='trinh_do_id', ascending=True).reset_index(drop=True)
print(df_trinhdo)

   trinh_do_id                 Trinh_do
0            0         (Không xác định)
1            3                 Cao đẳng
2            4                  Đại học
3            5                  Thạc sĩ
4            6                  Tiến sĩ
5            7              Phó tiến sĩ
6            9             Trung học PT
7           10                   PGS.TS
8           11  Trung học chuyên nghiệp
9           12                    12/12


## Load data

### [Nếu cần] Clear bảng

In [25]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Trinh_do"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

### Load data vào bảng Dim

In [26]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Trinh_do (ID_trinh_do, Loai_trinh_do) 
                VALUES (?, ?)
                """
for index, row in df_trinhdo.iterrows():
    values = (row['trinh_do_id'], 
              row['Trinh_do'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Dim_Lop

## Đọc data từ SQL Server

In [27]:
query_Lop = "SELECT Ten_lop FROM Lop" # Đọc dữ liệu từ bảng Lop trong CSDL libol
df_lop_lop = pd.read_sql(query_Lop, conn_libol)
query_LopBandoc = "SELECT DISTINCT dbo.DecodeUTF8String(Lop) AS Lop FROM Ban_doc" # Đọc dữ liệu từ bảng Ban_doc trong CSDL libol
df_lop_bandoc = pd.read_sql(query_LopBandoc, conn_libol)
print(df_lop_lop)
print(df_lop_bandoc)

C:\Users\admin\AppData\Local\Temp\ipykernel_21708\675338709.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_lop_lop = pd.read_sql(query_Lop, conn_libol)
C:\Users\admin\AppData\Local\Temp\ipykernel_21708\675338709.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_lop_bandoc = pd.read_sql(query_LopBandoc, conn_libol)


       Ten_lop
0      191040B
1      191040A
2    19109CL1B
3    19109CL1A
4    19109CL2B
..         ...
597    211611B
598    211611A
599    211612A
600    211612B
601      21950

[602 rows x 1 columns]
            Lop
0        017031
1       001011C
2       042030A
3       057090A
4       117450A
...         ...
4259        709
4260   14151CLC
4261    181311A
4262  19143CL1B
4263      23950

[4264 rows x 1 columns]


## Xử lý data

In [28]:
# Tạo data_frame mới gộp các hàng dữ liệu từ 2 data_frame kia
df_data_lop = pd.DataFrame({"ID_lop": pd.concat([df_lop_lop["Ten_lop"], 
                                                  df_lop_bandoc["Lop"]], 
                                                  ignore_index=True)})
df_data_lop['ID_lop'] = df_data_lop['ID_lop'].str.upper() # In hoa hết các hàng dữ liệu
for j, row in df_data_lop.iterrows():
    ten_nhom = row["ID_lop"]
    if ((pd.isna(ten_nhom)) or # Kiểm tra none
        (ten_nhom == "") or
        (ten_nhom == "0") or
        (ten_nhom == "00") or
        (ten_nhom == "000")):  # Kiểm tra NaN
        df_data_lop.at[j, 'ID_lop'] = "0"
df_data_lop = df_data_lop.sort_values(by="ID_lop", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn
df_data_lop = df_data_lop.drop_duplicates().reset_index(drop=True) # xóa những hàng bị trùng nhau
df_data_lop['ID_khoa'] = "0" # cho ID_Khoa = 0 (Không rõ) vì hiện tại chưa có dữ liệu về lớp thuộc khoa nào 
print(df_data_lop)

        ID_lop ID_khoa
0            0       0
1       001011       0
2      001011A       0
3      001011C       0
4       001012       0
...        ...     ...
4256  XÂY DỰNG       0
4257   ÊN11021       0
4258  ÊN14010A       0
4259  ÊN2D02VD       0
4260      ĐIỆN       0

[4261 rows x 2 columns]


## Load data

### [Nếu cần] Clear bảng

In [29]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Lop"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

### Load data vào bảng Dim_Lop

In [30]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Lop (ID_lop, ID_khoa) 
                VALUES (?, ?)
                """
for index, row in df_data_lop.iterrows():
    values = (row['ID_lop'], 
              row['ID_khoa'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Dim_Nhom_ban_doc

## Đọc data từ SQL Server

In [31]:
# Đọc dữ liệu từ bảng Lop trong CSDL libol
query_Nhombandoc = "SELECT Nhom_ID, dbo.DecodeUTF8String(Ten_nhom) AS Ten_nhom FROM Nhom_ban_doc"
df_nhombandoc = pd.read_sql(query_Nhombandoc, conn_libol)
print(df_nhombandoc)

    Nhom_ID                  Ten_nhom
0         5                          
1         6         .Cán bộ công chức
2         9          MƯỢN & ĐỌC - SKV
3        10  Tốt nghiệp_Cộng Tác viên
4        12               Đọc tại chỗ
5        14                   MƯỢN GT
6        15             MƯỢN GT & SKV
7        16    Chưa tham gia khóa học
8        17    Con CB & CTV (Mượn GT)
9        18          HỌC VIÊN CAO HỌC
10       19    Khoa ĐT chất lượng cao
11       20         NHÓM NGOÀI TRƯỜNG
12       21       GIÁO TRÌNH QUÉT LỘN
13       22    NHÓM LÃNH ĐẠO, QUẢN LÝ
14       23         Nhóm ngoài trường
15       24      Nhóm ký công nợ (TN)
16       25           Nghiên cứu sinh


C:\Users\admin\AppData\Local\Temp\ipykernel_21708\1031425255.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_nhombandoc = pd.read_sql(query_Nhombandoc, conn_libol)


## Xử lý data

In [32]:
df_nhombandoc = df_nhombandoc.drop_duplicates(subset='Ten_nhom').reset_index(drop=True) # xóa những hàng có Ten_nhom bị trùng nhau

new_row = pd.DataFrame({'Nhom_ID': [0], 'Ten_nhom': ['(Không xác định)']}) # Tạo hàng dữ liệu giả lập cho nhóm không xác định
df_nhombandoc = pd.concat([df_nhombandoc, new_row], ignore_index=True) # Thêm vào dataframe

for j, row in df_nhombandoc.iterrows(): # quét qua dữ liệu từng hàng của cột Ten_nhom
    ten_nhom = row["Ten_nhom"]
    if pd.isna(ten_nhom) or ten_nhom == "":  # Kiểm tra NaN or None
        df_nhombandoc.at[j, 'Ten_nhom'] = "(Không xác định)"
    else:
        ten_nhom = ten_nhom[0].upper() + ten_nhom[1:].lower() # Chỉnh sửa chữ hoa chữ thường nếu chuỗi không rỗng
        df_nhombandoc.at[j, 'Ten_nhom'] = ten_nhom

df_nhombandoc = df_nhombandoc.sort_values(by='Nhom_ID', ascending=True).reset_index(drop=True) # sắp xếp lại cho dễ nhìn
print(df_nhombandoc)


    Nhom_ID                  Ten_nhom
0         0          (không xác định)
1         5          (Không xác định)
2         6         .cán bộ công chức
3         9          Mượn & đọc - skv
4        10  Tốt nghiệp_cộng tác viên
5        12               Đọc tại chỗ
6        14                   Mượn gt
7        15             Mượn gt & skv
8        16    Chưa tham gia khóa học
9        17    Con cb & ctv (mượn gt)
10       18          Học viên cao học
11       19    Khoa đt chất lượng cao
12       20         Nhóm ngoài trường
13       21       Giáo trình quét lộn
14       22    Nhóm lãnh đạo, quản lý
15       23         Nhóm ngoài trường
16       24      Nhóm ký công nợ (tn)
17       25           Nghiên cứu sinh


## Load data

### [Nếu cần] Clear bảng

In [33]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Nhom_ban_doc"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

### Load vào bảng DIM_Nhom_ban_doc

In [34]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Nhom_ban_doc (ID_nhom_ban_doc, Nhom_ban_doc) 
                VALUES (?, ?)
                """
for index, row in df_nhombandoc.iterrows():
    values = (row['Nhom_ID'], 
              row['Ten_nhom'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Dim_Nhom_nghanh_nghe

## Đọc data từ SQL Server

In [35]:
query_Nhomnghanhnghe = "SELECT ID, dbo.DecodeUTF8String(Ten_nhom) AS Ten_nhom FROM Nhom_nghanh_nghe"
df_nhomnghanhnge = pd.read_sql(query_Nhomnghanhnghe, conn_libol)
print(df_nhomnghanhnge)

C:\Users\admin\AppData\Local\Temp\ipykernel_21708\173999304.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_nhomnghanhnge = pd.read_sql(query_Nhomnghanhnghe, conn_libol)


      ID                  Ten_nhom
0      0          (Không xác định)
1      1                      Y tế
2      5       Công nhân viên chức
3      9         Điện tử - Tin học
4     11                 Tài chính
..   ...                       ...
248  291        Công nghệ vật liệu
249  292         Vật liệu xây dựng
250  293                      Luật
251  294         Sư phạm công nghệ
252  295  Kỹ thuật cơ khí động lực

[253 rows x 2 columns]


## Xử lý data

In [36]:
for j, row in df_nhomnghanhnge.iterrows(): 
    ten_nhom = row["Ten_nhom"]
    if pd.isna(ten_nhom) or ten_nhom == "":  # Kiểm tra none hoặc NaN
        df_nhombandoc.at[j, 'Ten_nhom'] = "(Không xác định)" 
df_nhomnghanhnge = df_nhomnghanhnge.drop_duplicates(subset='Ten_nhom').reset_index(drop=True) # xóa những hàng bị trùng nhau
print(df_nhomnghanhnge)

      ID                          Ten_nhom
0      0                  (Không xác định)
1      1                              Y tế
2      5               Công nhân viên chức
3      9                 Điện tử - Tin học
4     11                         Tài chính
..   ...                               ...
232  288  Logistic và Tài chính thương mại
233  292                 Vật liệu xây dựng
234  293                              Luật
235  294                 Sư phạm công nghệ
236  295          Kỹ thuật cơ khí động lực

[237 rows x 2 columns]


## Load data

### [Nếu cần] Clear bảng

In [37]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Nhom_nghanh_nghe"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

### Load vào bảng DIM_Nhom_nghanh_nghe

In [38]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Nhom_nghanh_nghe (ID_nhom_nghanh_nghe, 
                            Nhom_nghanh_nghe) 
                VALUES (?, ?)
                """
for index, row in df_nhomnghanhnge.iterrows():
    values = (row['ID'], 
              row['Ten_nhom'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Dim_Quoc_gia

## Đọc data từ SQL Server

In [43]:
query_Quocgia = "SELECT Ma_nuoc_ID, Ma_ISO,Ten_nuoc_ISO FROM Ten_nuoc" # Đọc dữ liệu từ bảng Lop trong CSDL libol
df_quocgia = pd.read_sql(query_Quocgia, conn_libol)
print(df_quocgia)

     Ma_nuoc_ID Ma_ISO         Ten_nuoc_ISO
0           190     TG                 Togo
1           191     TK              Tokelau
2           192     TO                Tonga
3           193     TT  Trinidad and Tobago
4           194     TN              Tunesia
..          ...    ...                  ...
231         186     SY                Syria
232         187     TW               Taiwan
233         188     TZ             Tanzania
234         189     TH             Thailand
235         209     VN              Vietnam

[236 rows x 3 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_21708\621875265.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_quocgia = pd.read_sql(query_Quocgia, conn_libol)


## Xử lý data

In [44]:
# Tạo hàng dữ liệu giả lập cho nhóm không xác định
new_row = pd.DataFrame({'Ma_nuoc_ID': [0], 'Ma_ISO': ['NO'], 'Ten_nuoc_ISO': ['(Không xác định)']}) 
df_quocgia = pd.concat([df_quocgia, new_row], ignore_index=True) # Thêm vào dataframe

df_quocgia = df_quocgia.drop_duplicates(subset='Ten_nuoc_ISO').reset_index(drop=True) # xóa những dữ liệu cột Ten_nuoc_ISO bị trùng nhau
df_quocgia = df_quocgia.sort_values(by="Ma_nuoc_ID", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn
print(df_quocgia)

     Ma_nuoc_ID Ma_ISO        Ten_nuoc_ISO
0             0     NO    (Không xác định)
1             1     AF         Afghanistan
2             2     AL             Albania
3             3     DZ             Algeria
4             4     AS      American Samoa
..          ...    ...                 ...
232         232     AJ          Azerbaijan
233         233     CI             Croatia
234         234     XV            Slovenia
235         235     BN  Bosnia-Hercegovina
236         236     XN           Macedonia

[237 rows x 3 columns]


## Load data

### [Nếu cần] Clear bảng

In [45]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Quoc_gia"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

### Load vào bảng DIM_Quoc_gia

In [46]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Quoc_gia (ID_quoc_gia, Ma_ISO, Ten_nuoc_ISO) 
                VALUES (?, ?, ?)
                """
for index, row in df_quocgia.iterrows():
    values = (row['Ma_nuoc_ID'], 
              row['Ma_ISO'],
              row['Ten_nuoc_ISO'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Dim_Vat_mang_tin

## Đọc data SQL Server

In [47]:
# Đọc dữ liệu từ bảng Lop trong CSDL libol
query_Vatmangtin = "SELECT Vat_mang_tin_ID, dbo.DecodeUTF8String(Ky_hieu) AS Ky_hieu, dbo.DecodeUTF8String(Vat_mang_tin) AS Vat_mang_tin FROM Vat_mang_tin"
df_vatmangtin = pd.read_sql(query_Vatmangtin, conn_libol)
print(df_vatmangtin)

    Vat_mang_tin_ID Ky_hieu          Vat_mang_tin
0                 1      MC             Microfilm
1                 2      MF            Microfiche
2                 3       G                  Giấy
3                 4      BT               Băng từ
4                 5      ĐT                Đĩa từ
5                 6      CD  Đĩa CDROM (đĩa Laze)
6                 7      VA    Vật liệu nghe nhìn
7                 8    giấy                  None
8                 9  Dia tu                  None
9                10      GH                  None
10               11       B                  None
11               12     Tan                  None
12               13   Ebook          Sách điện tử
13               14      VT                  None
14               15       A                  None


C:\Users\admin\AppData\Local\Temp\ipykernel_21708\2402062624.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_vatmangtin = pd.read_sql(query_Vatmangtin, conn_libol)


## Xử lý data

In [48]:
new_row = pd.DataFrame({'Vat_mang_tin_ID': [0],'Ky_hieu': ['None'], 'Vat_mang_tin': ['(Không xác định)']}) # Tạo hàng dữ liệu giả lập cho nhóm không xác định
df_vatmangtin = pd.concat([df_vatmangtin, new_row], ignore_index=True) # Thêm vào dataframe
df_vatmangtin = df_vatmangtin.sort_values(by="Vat_mang_tin_ID", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn
print(df_vatmangtin)

    Vat_mang_tin_ID Ky_hieu          Vat_mang_tin
0                 0    None      (Không xác định)
1                 1      MC             Microfilm
2                 2      MF            Microfiche
3                 3       G                  Giấy
4                 4      BT               Băng từ
5                 5      ĐT                Đĩa từ
6                 6      CD  Đĩa CDROM (đĩa Laze)
7                 7      VA    Vật liệu nghe nhìn
8                 8    giấy                  None
9                 9  Dia tu                  None
10               10      GH                  None
11               11       B                  None
12               12     Tan                  None
13               13   Ebook          Sách điện tử
14               14      VT                  None
15               15       A                  None


## Load data

### [Nếu cần] Clear bảng

In [49]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Vat_mang_tin"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

### Load vào bảng Dim_Vat_mang_tin

In [50]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Vat_mang_tin (ID_vat_mang_tin, Ky_hieu, Vat_mang_tin) 
                VALUES (?, ?, ?)
                """
for index, row in df_vatmangtin.iterrows():
    values = (row['Vat_mang_tin_ID'], 
              row['Ky_hieu'],
              row['Vat_mang_tin'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Dim_Dang_tai_lieu

## Đọc data

In [55]:
# Đọc dữ liệu từ bảng Lop trong CSDL libol
query_Dangtailieu = """SELECT Dang_tai_lieu_ID, 
                        dbo.DecodeUTF8String(Dang_tai_lieu) AS Dang_tai_lieu, 
                        dbo.DecodeUTF8String(Ky_hieu_tai_lieu) AS Ky_hieu_tai_lieu,
                        LoanPeriod,
                        Renewals,
                        RenewalPeriod,
                        TimeUnit,
                        Fee,
                        OverdueFine,
                        FixedFee
                        FROM Dang_tai_lieu
                        """
df_dangtailieu = pd.read_sql(query_Dangtailieu, conn_libol)
print(df_dangtailieu)

    Dang_tai_lieu_ID                                    Dang_tai_lieu  \
0                  1                     Sách, chuyên khảo, tuyển tập   
1                  2                                        Bài trích   
2                  3                                Luận án, luận văn   
3                  4  Báo cáo kết quả  nghiên cứu; tổng kết; khảo sát   
4                  5                                 Báo cáo hội nghị   
5                  6                               Catalô công nghiệp   
6                  7                                       Tiêu chuẩn   
7                  8                                         Sáng chế   
8                  9                                  ấn phẩm định kỳ   
9                 10                                             Phim   
10                11                              Bản đồ, sách bản đồ   
11                12                    Hình vẽ, bản vẽ, tranh ảnh,..   
12                13                     Tờ rời giớ

C:\Users\admin\AppData\Local\Temp\ipykernel_21708\3235927709.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dangtailieu = pd.read_sql(query_Dangtailieu, conn_libol)


## Xử lý data


In [56]:
new_row = pd.DataFrame({'Dang_tai_lieu_ID': [0],
                        'Dang_tai_lieu': ['(Không xác định)'],
                        'Ky_hieu_tai_lieu': ['None'],
                        'Fee': [0],
                        'OverdueFine': [0],
                        'FixedFee': ['False'],
                        }) # Tạo hàng dữ liệu giả lập cho nhóm không xác định
df_dangtailieu = pd.concat([df_dangtailieu, new_row], ignore_index=True) # Thêm vào dataframe

columns_to_check = ['LoanPeriod', 'Renewals', 'RenewalPeriod', 'TimeUnit']
for index, row in df_dangtailieu.iterrows():
    for col in columns_to_check:
        if pd.isna(row[col]):
            df_dangtailieu.at[index, col] = 0  # Thay NaN bằng 0

df_dangtailieu = df_dangtailieu.sort_values(by="Dang_tai_lieu_ID", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn           
print(df_dangtailieu)

    Dang_tai_lieu_ID                                    Dang_tai_lieu  \
0                  0                                 (Không xác định)   
1                  1                     Sách, chuyên khảo, tuyển tập   
2                  2                                        Bài trích   
3                  3                                Luận án, luận văn   
4                  4  Báo cáo kết quả  nghiên cứu; tổng kết; khảo sát   
5                  5                                 Báo cáo hội nghị   
6                  6                               Catalô công nghiệp   
7                  7                                       Tiêu chuẩn   
8                  8                                         Sáng chế   
9                  9                                  ấn phẩm định kỳ   
10                10                                             Phim   
11                11                              Bản đồ, sách bản đồ   
12                12                    Hình vẽ, bả

## Load data

### [Nếu cần] Clear bảng

In [57]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Dang_tai_lieu"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

### Load vào bảng DIM_Dang_tai_lieu

In [58]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Dang_tai_lieu (ID_dang_tai_lieu, Dang_tai_lieu, Ky_hieu_tai_lieu, LoanPeriod, Renewals, RenewalPeriod, TimeUnit, Fee, OverdueFine, FixedFee) 
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
               """
for index, row in df_dangtailieu.iterrows():
    values = (row['Dang_tai_lieu_ID'], 
                row['Dang_tai_lieu'],
                row['Ky_hieu_tai_lieu'],
                row['LoanPeriod'],
                row['Renewals'],
                row['RenewalPeriod'],
                row['TimeUnit'],
                row['Fee'],
                row['OverdueFine'],
                row['FixedFee'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Dim_Ten_form

## Đọc data từ SQL Server

In [59]:
# Đọc dữ liệu từ bảng Lop trong CSDL libol
query_Tenform = """SELECT ID, 
                        dbo.DecodeUTF8String(Ten_form) AS Ten_form, 
                        Nguoi_tao,
                        Ngay_tao,
                        Ngay_sua_cuoi
                        FROM Ten_Form
                        """
df_tenform = pd.read_sql(query_Tenform, conn_libol)
print(df_tenform)

    ID                         Ten_form      Nguoi_tao            Ngay_tao  \
0   37                    Sách (USMARC)  Administrator 2001-05-19 16:00:51   
1   38              Băng Video (USMARC)  Administrator 2001-05-31 16:58:22   
2   40                Âm thanh (USMARC)  Administrator 2001-06-02 08:52:53   
3   39            Tệp máy tính (USMARC)  Administrator 2001-05-31 17:34:27   
4   41                  Bản đồ (USMARC)  Administrator 2001-08-09 17:20:27   
5   42         Ấn phẩm định kỳ (USMARC)  Administrator 2001-08-10 08:46:06   
6   43                   Sách (rút gọn)  Administrator 2001-08-21 11:50:25   
7   81  Biên mục Tiêu chuẩn và Quy phạm         DHSPKT 2002-08-21 08:55:07   
8   67                  Bài báo(DHSPKT)  Administrator 2001-11-27 11:09:01   
9   68                        Bài trích  Administrator 2001-11-27 11:16:28   
10  85               Biên mục Bài trích  Administrator 2002-08-21 12:36:16   
11  82                    Biên mục Sách         DHSPKT 2002-08-2

C:\Users\admin\AppData\Local\Temp\ipykernel_21708\64601752.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tenform = pd.read_sql(query_Tenform, conn_libol)


## Xử lý data

In [60]:
df_tenform['Ngay_tao'] = df_tenform['Ngay_tao'].dt.strftime('%Y%m%d').astype(int)
df_tenform['Ngay_sua_cuoi'] = df_tenform['Ngay_sua_cuoi'].dt.strftime('%Y%m%d').astype(int)

new_row = pd.DataFrame({'ID': [0], # Tạo hàng dữ liệu giả lập cho form không xác định
                        'Ten_form': ['(Không xác định)'],
                        'Nguoi_tao': ['(Không xác định)'],
                        'Ngay_tao': [0],
                        'Ngay_sua_cuoi': [0]})
df_tenform = pd.concat([df_tenform, new_row], ignore_index=True) # Thêm vào dataframe

df_tenform = df_tenform.sort_values(by="ID", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn           
print(df_tenform)

    ID                         Ten_form         Nguoi_tao  Ngay_tao  \
0    0                 (Không xác định)  (Không xác định)         0   
1   37                    Sách (USMARC)     Administrator  20010519   
2   38              Băng Video (USMARC)     Administrator  20010531   
3   39            Tệp máy tính (USMARC)     Administrator  20010531   
4   40                Âm thanh (USMARC)     Administrator  20010602   
5   41                  Bản đồ (USMARC)     Administrator  20010809   
6   42         Ấn phẩm định kỳ (USMARC)     Administrator  20010810   
7   43                   Sách (rút gọn)     Administrator  20010821   
8   67                  Bài báo(DHSPKT)     Administrator  20011127   
9   68                        Bài trích     Administrator  20011127   
10  81  Biên mục Tiêu chuẩn và Quy phạm            DHSPKT  20020821   
11  82                    Biên mục Sách            DHSPKT  20020821   
12  83          Biên mục Báo và Tạp chí            DHSPKT  20020821   
13  84

## Load data

## [Nếu cần] Clear bảng

In [61]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Ten_form"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

### Load vào bảng DIM_Ten_form

In [62]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Ten_form (ID_form, Ten_form, Nguoi_tao, Ngay_tao, Ngay_sua_cuoi) 
                VALUES (?, ?, ?, ?, ?)
               """
for index, row in df_tenform.iterrows():
    values = (row['ID'], 
                row['Ten_form'],
                row['Nguoi_tao'],
                row['Ngay_tao'],
                row['Ngay_sua_cuoi'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Dim_Thu_vien

## Đọc data từ SQL Server 

In [93]:
# Đọc dữ liệu từ bảng Lop trong CSDL libol
query_Thuvien = """SELECT dbo.DecodeUTF8String(Ten_viet_tat) AS Ten_viet_tat,
                        dbo.DecodeUTF8String(Thu_vien) AS Thu_vien, 
                        dbo.DecodeUTF8String(Dia_chi) AS Dia_chi,
                        gia_tri,
                        LocalLib
                        FROM Thu_vien
                        """
df_thuvien = pd.read_sql(query_Thuvien, conn_libol)
print(df_thuvien)

                    Ten_viet_tat                            Thu_vien  \
0                         DHSPKT   Thư viện Đại học Sư Phạm Kĩ Thuật   
1                         ĐHSPKT                                None   
2                        SDHSPKT                                None   
3                    ĐHSPKT##Vie                                None   
4                            Vie                                None   
5                    DHSPKT##Vie                                None   
6                    D9HSP T.HCM                                None   
7                         SPDHKT                                None   
8                  ĐHSPKT TP.HCM                                None   
9                          ĐSPKT                                None   
10                        HCMUTE                                None   
11                           DLC                                None   
12  TVTTHCM|bvie|cTVTTHCM|eAACR2                                

C:\Users\admin\AppData\Local\Temp\ipykernel_21708\133294711.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_thuvien = pd.read_sql(query_Thuvien, conn_libol)


## Xử lý data

In [94]:
new_row = pd.DataFrame({'Ten_viet_tat': ['0'], # Tạo hàng dữ liệu giả lập cho thư viện không xác định
                        'Thu_vien': ['(Không xác định)'],
                        'Dia_chi': ['(Không xác định)'], 
                        'gia_tri': ['(Không xác định)'],
                        'LocalLib': ['False']})
df_thuvien = pd.concat([df_thuvien, new_row], ignore_index=True) # Thêm vào dataframe

df_thuvien = df_thuvien.sort_values(by="Ten_viet_tat", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn           
print(df_thuvien)

                    Ten_viet_tat                            Thu_vien  \
0                              0                    (Không xác định)   
1                            CUD                                None   
2                    D9HSP T.HCM                                None   
3                         DHSPKT   Thư viện Đại học Sư Phạm Kĩ Thuật   
4                    DHSPKT##Vie                                None   
5                            DLC                                None   
6                       DNLM/DLC                                None   
7                            FQG                                None   
8                         HCMUTE                                None   
9                            RRR                                None   
10                           RVE                                None   
11                       SDHSPKT                                None   
12                         SINUS                                

## Load data

### [Nếu cần] Clear bảng

In [95]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Thu_vien"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

### load vào bảng DIM_Thu_vien

In [96]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Thu_vien (ID_thu_vien, Thu_vien, Dia_chi, Gia_tri, LocalLib) 
                VALUES (?, ?, ?, ?, ?)
               """
for index, row in df_thuvien.iterrows():
    values = (row['Ten_viet_tat'], 
                row['Thu_vien'],
                row['Dia_chi'],
                row['gia_tri'],
                row['LocalLib'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Dim_Kho

## Đọc data từ SQL Server

In [109]:
# Đọc dữ liệu từ bảng Lop trong CSDL libol
query_Kho= """SELECT Kho_ID, dbo.DecodeUTF8String(Kho) AS Kho, Thu_vien_ID, MaxID, Mo FROM Kho """
df_kho = pd.read_sql(query_Kho, conn_libol)

query_Thuvien = """SELECT Thu_vien_ID, dbo.DecodeUTF8String(Ten_viet_tat) AS Ten_viet_tat FROM Thu_vien """
df_thuvien = pd.read_sql(query_Thuvien, conn_libol)

print(df_kho)
print(df_thuvien)

   Kho_ID           Kho  Thu_vien_ID  MaxID     Mo
0       7            KM            2      2   True
1       8            NV            1  71094   True
2       9            GT            1  60109   True
3       5            KM            1  14805   True
4       6            KD            1  10574  False
5      19   Đọc tại chỗ            1      1  False
6      13  Kho thanh lý            1      1  False
7      14   Unavailbale            1      1  False
8      16           CLC            1    104  False
9      18       Kho Lưu            1      4  False
    Thu_vien_ID                  Ten_viet_tat
0             1                        DHSPKT
1             2                        ĐHSPKT
2             3                       SDHSPKT
3             4                   ĐHSPKT##Vie
4             5                           Vie
5             6                   DHSPKT##Vie
6             7                   D9HSP T.HCM
7             8                        SPDHKT
8             9          

C:\Users\admin\AppData\Local\Temp\ipykernel_21708\2520694761.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_kho = pd.read_sql(query_Kho, conn_libol)
C:\Users\admin\AppData\Local\Temp\ipykernel_21708\2520694761.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_thuvien = pd.read_sql(query_Thuvien, conn_libol)


## Xử lý data

In [110]:
map_dict = dict(zip(df_thuvien['Thu_vien_ID'], df_thuvien['Ten_viet_tat']))

for index, row in df_kho.iterrows():
    id_thu_vien = row['Thu_vien_ID']
    if id_thu_vien in map_dict:
        # Thay thế ID_Thu_vien bằng Ten_viet_tat nếu khớp
        df_kho.at[index, 'Thu_vien_ID'] = map_dict[id_thu_vien]

new_row = pd.DataFrame({'Kho_ID': [0], # Tạo hàng dữ liệu giả lập cho kho không xác định
                        'Kho': ['(Không xác định)'],
                        'Thu_vien_ID': ['0'],
                        'MaxID': [0], 
                        'Mo': ['False']})
df_kho = pd.concat([df_kho, new_row], ignore_index=True) # Thêm vào dataframe

df_kho = df_kho.sort_values(by="Kho_ID", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn           
print(df_kho)

    Kho_ID               Kho Thu_vien_ID  MaxID     Mo
0        0  (Không xác định)           0      0  False
1        5                KM      DHSPKT  14805   True
2        6                KD      DHSPKT  10574  False
3        7                KM      ĐHSPKT      2   True
4        8                NV      DHSPKT  71094   True
5        9                GT      DHSPKT  60109   True
6       13      Kho thanh lý      DHSPKT      1  False
7       14       Unavailbale      DHSPKT      1  False
8       16               CLC      DHSPKT    104  False
9       18           Kho Lưu      DHSPKT      4  False
10      19       Đọc tại chỗ      DHSPKT      1  False


C:\Users\admin\AppData\Local\Temp\ipykernel_21708\1537312163.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'ĐHSPKT' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_kho.at[index, 'Thu_vien_ID'] = map_dict[id_thu_vien]


## Load data

### [Nếu cần] Clear bảng

In [111]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Kho"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

### Load vào bảng DIM_Kho

In [112]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Kho (ID_kho, Kho, ID_thu_vien, MaxID, Mo) 
                VALUES (?, ?, ?, ?, ?)
               """
for index, row in df_kho.iterrows():
    values = (row['Kho_ID'], 
                row['Kho'],
                row['Thu_vien_ID'],
                row['MaxID'],
                row['Mo'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Dim_Ban_doc

## Đọc data từ SQL Server

In [160]:
query_Bandoc = """
SELECT TOP (100) So_the,
       dbo.DecodeUTF8String(Ho_ten) AS Ho_ten,
       Ngay_sinh,
       Dan_toc_ID,
       Trinh_do_ID,
       dbo.DecodeUTF8String(So_dien_thoai) AS So_dien_thoai,
       dbo.DecodeUTF8String(Nghe_nghiep) AS Nghe_nghiep,
       dbo.DecodeUTF8String(Co_quan) AS Co_quan,
       dbo.DecodeUTF8String(chuc_vu) AS chuc_vu,
       dbo.DecodeUTF8String(Dia_chi) AS Dia_chi,
       dbo.DecodeUTF8String(Dia_chi_thuong_tru) AS Dia_chi_thuong_tru,
       ID_Khoa_hoc,
       Lop,
       Anh,
       Ngay_cap,
       Ngay_het_han,
       Email,
       Nhom_ID,
       Nhom_nghanh_nghe_ID,
       Gioi_tinh,
       Status,
       dbo.DecodeUTF8String(Ghi_chu) AS Ghi_chu,
       Mat_khau
  FROM Ban_doc
  WHERE Year(Ngay_sinh) = 2003
"""
df_bandoc = pd.read_sql(query_Bandoc, conn_libol)
print(df_bandoc)


C:\Users\admin\AppData\Local\Temp\ipykernel_21708\2642061735.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_bandoc = pd.read_sql(query_Bandoc, conn_libol)


      So_the                  Ho_ten  Ngay_sinh  Dan_toc_ID  Trinh_do_ID  \
0   21104002           Bùi Huỳnh Anh 2003-08-19           1           12   
1   21104003      Cao Thị Nguyệt Ánh 2003-09-04           1           12   
2   21104005        Lê Thị Hồng Châu 2003-10-26           1           12   
3   21104007       Trương Quang Được 2003-10-17           1           12   
4   21104008          Hoàng Minh Đức 2003-08-20           1           12   
..       ...                     ...        ...         ...          ...   
95  21109062       Ngô Thị Trúc Ngân 2003-01-13           1           12   
96  21109063   Nguyễn Thị Thanh Ngân 2003-02-20           1           12   
97  21109064         Nguyễn Thế Ngọc 2003-09-18           1           12   
98  21109065  Nguyễn Thị Thảo Nguyên 2003-12-28           1           12   
99  21109066    Bùi Quang Thiên Nhật 2003-11-17           1           12   

   So_dien_thoai Nghe_nghiep Co_quan chuc_vu Dia_chi  ...                Anh  \
0     0

## Xử lý data

### Xử lý data rỗng hoặc " "

In [161]:
df_bandoc = df_bandoc.replace('', None)
print(df_bandoc[['Nghe_nghiep', 'Co_quan', 'chuc_vu']])

   Nghe_nghiep Co_quan chuc_vu
0         None    None    None
1         None    None    None
2         None    None    None
3         None    None    None
4         None    None    None
..         ...     ...     ...
95        None    None    None
96        None    None    None
97        None    None    None
98        None    None    None
99        None    None    None

[100 rows x 3 columns]


### Xử lý kiểu date

In [162]:
query_date = "SELECT Date_key FROM DIM_date"
df_date = pd.read_sql(query_date, conn_dwh_lib)

date_ids = set(df_date['Date_key'])

# chuyển date về dang int 
# kiểm tra nhưng ngày đó có tồn tại trong date_key của bảng DIM_date hay không ?
df_bandoc['Ngay_sinh'] = df_bandoc['Ngay_sinh'].apply(lambda x: int(x.strftime('%Y%m%d')) if pd.notna(x) and int(x.strftime('%Y%m%d')) in date_ids else 0)
df_bandoc['Ngay_cap'] = df_bandoc['Ngay_cap'].apply(lambda x: int(x.strftime('%Y%m%d')) if pd.notna(x) and int(x.strftime('%Y%m%d')) in date_ids else 0)
df_bandoc['Ngay_het_han'] = df_bandoc['Ngay_het_han'].apply(lambda x: int(x.strftime('%Y%m%d')) if pd.notna(x) and int(x.strftime('%Y%m%d')) in date_ids else 0)

print(df_bandoc[['Ngay_sinh', 'Ngay_cap', 'Ngay_het_han']])

    Ngay_sinh  Ngay_cap  Ngay_het_han
0    20030819  20211031      20250815
1    20030904  20211031      20250821
2    20031026  20211031      20251009
3    20031017  20211031      20250926
4    20030820  20211031      20241106
..        ...       ...           ...
95   20030113  20211031      20240710
96   20030220  20211031      20251002
97   20030918  20211031      20241020
98   20031228  20211031      20240313
99   20031117  20211031      20221031

[100 rows x 3 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_21708\2258721904.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_date = pd.read_sql(query_date, conn_dwh_lib)


### Xử lý Dan_toc

In [163]:
# đọc dữ liệu lấy từ bộ về
df_Data_Dim_Dan_toc = pd.read_csv("./Source-Data/Data_Dan_toc.csv")
df_Data_Dim_Dan_toc = df_Data_Dim_Dan_toc.where(pd.notnull(df_Data_Dim_Dan_toc), None)
# đọc dữ liệu đã lưu trong sql server
query_Dan_toc = "SELECT Id, dbo.DecodeUTF8String(Dan_toc) AS Dan_toc FROM Dan_toc "
df_Dan_toc = pd.read_sql(query_Dan_toc, conn_libol)

#tìm và mapping 2 bảng lại
df_mapping = df_Dan_toc.copy()
df_mapping['Mapping Mã'] = None
df_mapping['CSV Mã'] = df_Data_Dim_Dan_toc['Mã']
df_mapping['CSV Tên'] = df_Data_Dim_Dan_toc['Tên']
df_mapping['CSV Tên khác'] = df_Data_Dim_Dan_toc['Tên gọi khác'].str.lower()
for i, dan_toc in enumerate(df_mapping['Dan_toc']):
    dan_toc = dan_toc.lower()
    dan_toc_bogach = dan_toc.replace("-", " ")
    dan_toc_botrong = dan_toc.replace(" ", "-")

    for j, row in df_mapping.iterrows():
        CSV_ten = row['CSV Tên'].lower() if pd.notna(row['CSV Tên']) else ""
        CSV_ten_khac = row['CSV Tên khác'].lower() if pd.notna(row['CSV Tên khác']) else ""
        if ((pd.notna(CSV_ten) and dan_toc == CSV_ten) or 
            (pd.notna(CSV_ten_khac) and dan_toc in CSV_ten_khac) or 
            (pd.notna(CSV_ten) and dan_toc_bogach == CSV_ten) or 
            (pd.notna(CSV_ten_khac) and dan_toc_bogach in CSV_ten_khac) or
            (pd.notna(CSV_ten) and dan_toc_botrong == CSV_ten) or 
            (pd.notna(CSV_ten_khac) and dan_toc_botrong in CSV_ten_khac)):
            df_mapping.at[i, 'Mapping Mã'] = row['CSV Mã']
            break
        else:
            df_mapping.at[i, 'Mapping Mã'] = 56
print(df_mapping)

    Id  Dan_toc Mapping Mã  CSV Mã CSV Tên  \
0    1     Kinh        1.0     1.0    Kinh   
1    2    Mường        3.0     2.0     Tày   
2    3      Tày        2.0     3.0    Thái   
3    4     Thái        3.0     4.0     Hoa   
4    5      Hoa        4.0     5.0  Khơ-me   
..  ..      ...        ...     ...     ...   
68  71     Ê Đê       12.0     NaN     NaN   
69  72      Thổ        2.0     NaN     NaN   
70  73    Kờ Ho         56     NaN     NaN   
71  74     Jrai         56     NaN     NaN   
72  75  Châu mạ       28.0     NaN     NaN   

                                         CSV Tên khác  
0                                                việt  
1           thổ, ngạn, phén, thù lao, pa dí, tày khao  
2   tày đăm, tày mười, tày thanh, mán thanh, hàng ...  
3   hán, triều châu, phúc kiến, quảng đông, hải na...  
4              cur, cul, cu, thổ, việt gốc miên, krôm  
..                                                ...  
68                                                NaN  

C:\Users\admin\AppData\Local\Temp\ipykernel_21708\800806833.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_Dan_toc = pd.read_sql(query_Dan_toc, conn_libol)


In [164]:
map_dict = dict(zip(df_mapping['Id'], df_mapping['Mapping Mã'])) # Tạo map_dict để ánh xạ từ Id sang Mapping_Ma trong df_mapping

for index, row in df_bandoc.iterrows(): # Lặp qua từng dòng trong df_ban_doc để cập nhật Dan_toc_ID
    dan_toc_id = row['Dan_toc_ID']
    
    if pd.isna(dan_toc_id): # Kiểm tra nếu Dan_toc_ID rỗng (None hoặc NaN)
        df_bandoc.at[index, 'Dan_toc_ID'] = 0
    else:
        if dan_toc_id in map_dict: # Kiểm tra nếu Dan_toc_ID có trong map_dict
            df_bandoc.at[index, 'Dan_toc_ID'] = map_dict[dan_toc_id]# Nếu tìm thấy, thay thế bằng giá trị Mapping_Ma
        else:
            df_bandoc.at[index, 'Dan_toc_ID'] = 0 # Nếu không tìm thấy, gán Dan_toc_ID bằng 0
print(df_bandoc['Dan_toc_ID'])


0     1
1     1
2     1
3     1
4     1
     ..
95    1
96    1
97    1
98    1
99    1
Name: Dan_toc_ID, Length: 100, dtype: int64


### Xử lý Trinh_do

In [165]:
query_Trinhdo = "SELECT ID_trinh_do FROM DIM_Trinh_do"
df_trinhdo = pd.read_sql(query_Trinhdo, conn_dwh_lib)

trinhdo_ids = set(df_trinhdo['ID_trinh_do'])

# Duyệt qua từng giá trị Lop trong df_bandoc và kiểm tra xem nó có nằm trong trinhdo_ids hay không. 
# Nếu có thì dữ nguyên nếu không, gán giá trị bằng 0.
df_bandoc['Trinh_do_ID'] = df_bandoc['Trinh_do_ID'].apply(lambda x: x if x in trinhdo_ids else 0)

print(df_bandoc['Trinh_do_ID'])


0     12
1     12
2     12
3     12
4     12
      ..
95    12
96    12
97    12
98    12
99    12
Name: Trinh_do_ID, Length: 100, dtype: int64


C:\Users\admin\AppData\Local\Temp\ipykernel_21708\3494513940.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_trinhdo = pd.read_sql(query_Trinhdo, conn_dwh_lib)


### Xử lý Lop


In [166]:
query_lop = "SELECT ID_lop FROM DIM_Lop"
df_lop = pd.read_sql(query_lop, conn_dwh_lib)

lop_ids = set(df_lop['ID_lop'])

# Duyệt qua từng giá trị Lop trong df_bandoc và kiểm tra xem nó có nằm trong lop_ids hay không. 
# Nếu có, chuyển sang dạng chữ in hoa (upper()), nếu không, gán giá trị bằng 0.
df_bandoc['Lop'] = df_bandoc['Lop'].str.upper()
df_bandoc['Lop'] = df_bandoc['Lop'].apply(lambda x: x.upper() if x in lop_ids else 0)

print(df_bandoc['Lop'])

0      21104C
1      21104B
2      21104B
3      21104A
4      21104A
       ...   
95    211091B
96    211092B
97    211092A
98    211091A
99    211092B
Name: Lop, Length: 100, dtype: object


C:\Users\admin\AppData\Local\Temp\ipykernel_21708\783121078.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_lop = pd.read_sql(query_lop, conn_dwh_lib)


### Xử lý Nhom_ban_doc

In [167]:
query_Nhombandoc = "SELECT ID_nhom_ban_doc FROM DIM_Nhom_ban_doc"
df_nhombandoc = pd.read_sql(query_Nhombandoc, conn_dwh_lib)

nhombandoc_ids = set(df_nhombandoc['ID_nhom_ban_doc'])

# Duyệt qua từng giá trị Lop trong df_bandoc và kiểm tra xem nó có nằm trong nhombandoc_ids hay không. 
# Nếu có thì dữ nguyên nếu không, gán giá trị bằng 0.
df_bandoc['Nhom_ID'] = df_bandoc['Nhom_ID'].apply(lambda x: x if x in nhombandoc_ids else 0)

print(df_bandoc['Nhom_ID'])

0     15
1     15
2     15
3     15
4     15
      ..
95    15
96    15
97    15
98    15
99    15
Name: Nhom_ID, Length: 100, dtype: int64


C:\Users\admin\AppData\Local\Temp\ipykernel_21708\3967328555.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_nhombandoc = pd.read_sql(query_Nhombandoc, conn_dwh_lib)


### Xử lý Nhom_nghanh_nghe

In [168]:
query_Nhomnghanhnghe = "SELECT ID_nhom_nghanh_nghe FROM DIM_Nhom_nghanh_nghe"
df_nhomnghanhnge = pd.read_sql(query_Nhomnghanhnghe, conn_dwh_lib)

nhomnghanhnghe_ids = set(df_nhomnghanhnge['ID_nhom_nghanh_nghe'])

# Duyệt qua từng giá trị Lop trong df_bandoc và kiểm tra xem nó có nằm trong nhomnghanhnghe_ids hay không. 
# Nếu có thì dữ nguyên nếu không, gán giá trị bằng 0.
df_bandoc['Nhom_nghanh_nghe_ID'] = df_bandoc['Nhom_nghanh_nghe_ID'].apply(lambda x: x if x in nhomnghanhnghe_ids else 0)

print(df_bandoc['Nhom_nghanh_nghe_ID'])

0     0
1     0
2     0
3     0
4     0
     ..
95    0
96    0
97    0
98    0
99    0
Name: Nhom_nghanh_nghe_ID, Length: 100, dtype: int64


C:\Users\admin\AppData\Local\Temp\ipykernel_21708\4291431786.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_nhomnghanhnge = pd.read_sql(query_Nhomnghanhnghe, conn_dwh_lib)


## Load data

### [Nếu cần] Clear bảng

In [169]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Ban_doc"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

### Load vào bảng DIM_Ban_doc

In [170]:
# Tạo cursor để thao tác với cơ sở dữ liệu
cursor_dwh = conn_dwh_lib.cursor()

# Chuẩn bị câu lệnh chèn dữ liệu
insert_query = """
                INSERT INTO DIM_Ban_doc (
                    ID_ban_doc, Ho_ten, Ngay_sinh, ID_dan_toc, ID_trinh_do,
                    So_dien_thoai, Nghe_nghiep, Co_quan, Chuc_vu,
                    Dia_chi_tam_tru, Dia_chi_thuong_tru, ID_khoa_hoc, ID_lop,
                    Anh, Ngay_cap, Ngay_het_han, Email, ID_nhom_ban_doc,
                    ID_nhom_nghanh_nghe, Gioi_tinh, Tinh_trang, Ghi_chu, Mat_khau
                ) 
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
               """

# Chuyển đổi dữ liệu từ DataFrame thành danh sách các tuple để chèn
data_to_insert = [
    (
        row['So_the'], row['Ho_ten'], row['Ngay_sinh'], row['Dan_toc_ID'], row['Trinh_do_ID'],
        row['So_dien_thoai'], row['Nghe_nghiep'], row['Co_quan'], row['chuc_vu'],
        row['Dia_chi'], row['Dia_chi_thuong_tru'], row['ID_Khoa_hoc'], row['Lop'],
        row['Anh'], row['Ngay_cap'], row['Ngay_het_han'], row['Email'], row['Nhom_ID'],
        row['Nhom_nghanh_nghe_ID'], row['Gioi_tinh'], row['Status'], row['Ghi_chu'], row['Mat_khau']
    )
    for index, row in df_bandoc.iterrows()
]

# Sử dụng executemany để chèn dữ liệu cùng lúc
cursor_dwh.executemany(insert_query, data_to_insert)

# Commit thay đổi
conn_dwh_lib.commit()

# Đóng cursor và kết nối
cursor_dwh.close()
conn_dwh_lib.close()


# ETL bảng DIM_Tai_lieu

## Đọc data từ SQL 

In [29]:
query_Tailieu = """
SELECT TOP (100) Tai_lieu_ID,
       Ma_tai_lieu,
       dbo.DecodeUTF8String(Nguoi_nhap_tin) AS Nguoi_nhap_tin,
       dbo.DecodeUTF8String(Nguoi_kiem_tra) AS Nguoi_kiem_tra,
       Nuoc_cung_cap_ID,
       Co_quan_cung_cap_ID,
       Ngay_giao_dich,
       Cap_mo_ta_thu_muc,
       Muc_do_mat,
       Vat_mang_tin_ID,
       Dang_tai_lieu_ID,
       Kieu_ban_ghi,
       Form_ID,
       Leader,
       OPAC,
       Bieu_ghi_nhap_hoi_co,
      dbo.DecodeUTF8String(CallNumber) AS CallNumber
  FROM Tai_lieu
  WHERE YEAR(Ngay_giao_dich) = 2024
"""
df_tailieu = pd.read_sql(query_Tailieu, conn_libol)
print(df_tailieu)

C:\Users\admin\AppData\Local\Temp\ipykernel_12232\799213490.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tailieu = pd.read_sql(query_Tailieu, conn_libol)


    Tai_lieu_ID  Ma_tai_lieu      Nguoi_nhap_tin      Nguoi_kiem_tra  \
0           418  SK020000429     Nguyễn Anh Khoa                Điền   
1           981  SK020000998     Nguyễn Anh Khoa      Phạm Minh Quân   
2          1009  SK020001027     Nguyễn Anh Khoa      Phạm Minh Quân   
3          1083  SK020001101       Vũ Trọng Luật      Phạm Minh Quân   
4          1156  SK020001177     Nguyễn Anh Khoa      Phạm Minh Quân   
..          ...          ...                 ...                 ...   
95        20612  SK070020613     Hồ Thị Thu Hoài  Nguyễn Thanh Giang   
96        21178  SK070021184  Nguyễn Thanh Giang  Nguyễn Thanh Giang   
97        21342  SK070021343     Hồ Thị Thu Hoài                       
98        21469  SK070021471     Hồ Thị Thu Hoài  Nguyễn Thanh Giang   
99        21793  SK070021794     Hồ Thị Thu Hoài  Nguyễn Thanh Giang   

   Nuoc_cung_cap_ID  Co_quan_cung_cap_ID      Ngay_giao_dich  \
0              None                  1.0 2024-10-25 15:43:00   
1      

## Xử lý data

### Xử lý NaN và " "

In [30]:
df_tailieu = df_tailieu.replace('', None)
df_tailieu = df_tailieu.replace(np.nan, None)
print(df_tailieu)

    Tai_lieu_ID  Ma_tai_lieu      Nguoi_nhap_tin      Nguoi_kiem_tra  \
0           418  SK020000429     Nguyễn Anh Khoa                Điền   
1           981  SK020000998     Nguyễn Anh Khoa      Phạm Minh Quân   
2          1009  SK020001027     Nguyễn Anh Khoa      Phạm Minh Quân   
3          1083  SK020001101       Vũ Trọng Luật      Phạm Minh Quân   
4          1156  SK020001177     Nguyễn Anh Khoa      Phạm Minh Quân   
..          ...          ...                 ...                 ...   
95        20612  SK070020613     Hồ Thị Thu Hoài  Nguyễn Thanh Giang   
96        21178  SK070021184  Nguyễn Thanh Giang  Nguyễn Thanh Giang   
97        21342  SK070021343     Hồ Thị Thu Hoài                None   
98        21469  SK070021471     Hồ Thị Thu Hoài  Nguyễn Thanh Giang   
99        21793  SK070021794     Hồ Thị Thu Hoài  Nguyễn Thanh Giang   

   Nuoc_cung_cap_ID Co_quan_cung_cap_ID      Ngay_giao_dich Cap_mo_ta_thu_muc  \
0              None                 1.0 2024-10-25 15:

### Xử lý kiểu date

In [31]:
query_date = "SELECT Date_key FROM DIM_Date"
df_date = pd.read_sql(query_date, conn_dwh_lib)
date_ids = set(df_date['Date_key'])
# chuyển date về dang int 
# kiểm tra nhưng ngày đó có tồn tại trong date_key của bảng DIM_date hay không ?
df_tailieu['Ngay_giao_dich'] = df_tailieu['Ngay_giao_dich'].apply(lambda x: int(x.strftime('%Y%m%d')) if pd.notna(x) and int(x.strftime('%Y%m%d')) in date_ids else 0)
print(df_tailieu[['Ngay_giao_dich']])

    Ngay_giao_dich
0         20241025
1         20240125
2         20240125
3         20240125
4         20240125
..             ...
95        20240416
96        20240325
97        20240909
98        20240125
99        20240125

[100 rows x 1 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_12232\1526097435.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_date = pd.read_sql(query_date, conn_dwh_lib)


### Xử lý ID_quoc_gia

In [32]:
query_Quocgia = "SELECT ID_quoc_gia FROM DIM_Quoc_gia"
df_quocgia = pd.read_sql(query_Quocgia, conn_dwh_lib)
quocgia_ids = set(df_quocgia['ID_quoc_gia'])
# chuyển date về dang int 
# kiểm tra nhưng ngày đó có tồn tại trong ID_quoc_gia của bảng DIM_Quoc_gia hay không ?
df_tailieu['Nuoc_cung_cap_ID'] = df_tailieu['Nuoc_cung_cap_ID'].apply(lambda x: x if pd.notna(x) and x in quocgia_ids else 0)
print(df_tailieu[['Nuoc_cung_cap_ID']])

    Nuoc_cung_cap_ID
0                  0
1                  0
2                  0
3                  0
4                  0
..               ...
95                 0
96                 0
97                 0
98                 0
99                 0

[100 rows x 1 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_12232\2869368773.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_quocgia = pd.read_sql(query_Quocgia, conn_dwh_lib)


### Xử lý ID_vat_mang_tin

In [33]:
query_Vatmangtin = "SELECT ID_vat_mang_tin FROM DIM_Vat_mang_tin"
df_vatmangtin = pd.read_sql(query_Vatmangtin, conn_dwh_lib)
vatmangtin_ids = set(df_vatmangtin['ID_vat_mang_tin'])
# chuyển date về dang int
# kiểm tra nhưng ngày đó có tồn tại trong ID_vat_mang_tin của bảng DIM_Vat_mang_tin hay không ?
df_tailieu['Vat_mang_tin_ID'] = df_tailieu['Vat_mang_tin_ID'].apply(lambda x: x if pd.notna(x) and x in vatmangtin_ids else 0)
print(df_tailieu[['Vat_mang_tin_ID']])

    Vat_mang_tin_ID
0                 3
1                 3
2                 3
3                 3
4                 3
..              ...
95                3
96                3
97                3
98                3
99                3

[100 rows x 1 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_12232\272293049.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_vatmangtin = pd.read_sql(query_Vatmangtin, conn_dwh_lib)


### Xử lý ID_dang_tai_lieu

In [34]:
query_Dangtailieu = "SELECT ID_dang_tai_lieu FROM DIM_Dang_tai_lieu"
df_dangtailieu = pd.read_sql(query_Dangtailieu, conn_dwh_lib)
dangtailieu_ids = set(df_dangtailieu['ID_dang_tai_lieu'])
# chuyển date về dang int
# kiểm tra nhưng ngày đó có tồn tại trong ID_dang_tai_lieu của bảng DIM_Dang_tai_lieu hay không ?
df_tailieu['Dang_tai_lieu_ID'] = df_tailieu['Dang_tai_lieu_ID'].apply(lambda x: x if pd.notna(x) and x in dangtailieu_ids else 0)
print(df_tailieu[['Dang_tai_lieu_ID']])

    Dang_tai_lieu_ID
0                  1
1                  1
2                  1
3                  1
4                  1
..               ...
95                 1
96                 1
97                 1
98                 1
99                 1

[100 rows x 1 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_12232\1936184778.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dangtailieu = pd.read_sql(query_Dangtailieu, conn_dwh_lib)


### Xử lý ID_form

In [35]:
query_form = "SELECT ID_form FROM DIM_Ten_form"
df_tenform = pd.read_sql(query_form, conn_dwh_lib)
tenform_ids = set(df_tenform['ID_form'])
# chuyển date về dang int
# kiểm tra nhưng ngày đó có tồn tại trong ID_dang_tai_lieu của bảng DIM_Dang_tai_lieu hay không ?
df_tailieu['Form_ID'] = df_tailieu['Form_ID'].apply(lambda x: x if pd.notna(x) and x in tenform_ids else 0)
print(df_tailieu[['Form_ID']])

    Form_ID
0        83
1        83
2        83
3        83
4        83
..      ...
95       83
96       83
97       83
98       83
99       83

[100 rows x 1 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_12232\380232656.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tenform = pd.read_sql(query_form, conn_dwh_lib)


## Load data

### [Nếu cần] Clear bảng

In [40]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Tai_lieu"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

### Load vào bảng DIM_Tai_lieu

In [42]:
# Tạo cursor để thao tác với cơ sở dữ liệu
cursor_dwh = conn_dwh_lib.cursor()

# Chuẩn bị câu lệnh chèn dữ liệu
insert_query = """
                INSERT INTO DIM_Tai_lieu (
                    ID_tai_lieu, Ma_tai_lieu, 
                    Nguoi_nhap_tin, Nguoi_kiem_tra, 
                    ID_quoc_gia, ID_co_quan_cung_cap,
                    Ngay_giao_dich,
                    Cap_mo_ta_thu_muc, Muc_do_mat, 
                    ID_vat_mang_tin, ID_dang_tai_lieu,
                    Kieu_ban_ghi, ID_form, 
                    Leader, OPAC, Bieu_ghi_nhap_hoi_co, CallNumber
                ) 
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
               """
# Chuyển đổi dữ liệu từ DataFrame thành danh sách các tuple để chèn
data_to_insert = [
    (
        row['Tai_lieu_ID'], row['Ma_tai_lieu'],
        row['Nguoi_nhap_tin'], row['Nguoi_kiem_tra'],
        row['Nuoc_cung_cap_ID'], row['Co_quan_cung_cap_ID'],
        row['Ngay_giao_dich'], 
        row['Cap_mo_ta_thu_muc'], row['Muc_do_mat'],
        row['Vat_mang_tin_ID'], row['Dang_tai_lieu_ID'], 
        row['Kieu_ban_ghi'], row['Form_ID'],
        row['Leader'], row['OPAC'], row['Bieu_ghi_nhap_hoi_co'], row['CallNumber'], 
    )
    for index, row in df_tailieu.iterrows()
]

# Sử dụng executemany để chèn dữ liệu cùng lúc
cursor_dwh.executemany(insert_query, data_to_insert)

# Commit thay đổi
conn_dwh_lib.commit()

# Đóng cursor và kết nối
cursor_dwh.close()
conn_dwh_lib.close()